In [18]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 加载数据
df = pd.read_csv("C:/Users/31314/Desktop/最终面板数据.csv")  # 请替换为您的实际数据加载方式

# 重命名列以便使用
df = df.rename(columns={
    'Mkt-RF': 'MKT_RF',
    'Return': 'ret'  # 确保股票收益率列名为ret
})

# 检查必需列
print("数据列名:", df.columns.tolist())
print("数据形状:", df.shape)
print("\n前5行数据:")
print(df.head())

# 确保有必需的列
required_cols = ['ticker', 'ret', 't', 'MKT_RF', 'RF']
if 'SMB' in df.columns and 'HML' in df.columns:
    required_cols.extend(['SMB', 'HML'])
if 'MOM' in df.columns:
    required_cols.append('MOM')

# 计算超额收益
df['excess_ret'] = df['ret'] - df['RF']

# 定义窗口函数
def define_windows(t):
    """为每个观测点定义所属窗口"""
    if -260 <= t <= -11:
        return 'estimation'
    elif -1 <= t <= 1:
        return 'short_term'
    elif 0 <= t <= 5:
        return 'medium_term'
    elif -5 <= t <= 30:
        return 'extended'
    elif 0 <= t <= 60:
        return 'post_event'
    else:
        return 'other'

# 添加窗口列
df['window'] = df['t'].apply(define_windows)

# 创建新的列用于存储AR和CAR
df['AR_CAPM'] = np.nan
df['AR_FF4'] = np.nan
df['CAR_CAPM_short_term'] = np.nan
df['CAR_CAPM_medium_term'] = np.nan
df['CAR_CAPM_extended'] = np.nan
df['CAR_CAPM_post_event'] = np.nan
df['CAR_FF4_short_term'] = np.nan
df['CAR_FF4_medium_term'] = np.nan
df['CAR_FF4_extended'] = np.nan
df['CAR_FF4_post_event'] = np.nan
df['AR_st_CAPM'] = np.nan  # 标准化AR
df['AR_st_FF4'] = np.nan   # 标准化AR

print(f"\n窗口分布:")
print(df['window'].value_counts())

数据列名: ['date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'ret', 'event_id', 'company', 'event_date', 'ai_mentioned', 'calendar_diff', 't', 'MKT_RF', 'SMB', 'HML', 'RF']
数据形状: (104087, 18)

前5行数据:
         date   ticker  Open  High  Low  Close  Adj Close       ret  \
0  2021-07-14  0442.HK   0.7   0.7  0.7    0.7        0.7       NaN   
1  2021-07-15  0442.HK   0.7   0.7  0.7    0.7        0.7  0.000000   
2  2021-07-16  0442.HK   0.8   0.8  0.8    0.8        0.8  0.133531   
3  2021-07-19  0442.HK   0.8   0.8  0.8    0.8        0.8  0.000000   
4  2021-07-20  0442.HK   0.8   0.8  0.8    0.8        0.8  0.000000   

             event_id company event_date  ai_mentioned  calendar_diff    t  \
0  0442.HK_2022-05-10    Doma  2022/5/10             1           -300 -201   
1  0442.HK_2022-05-10    Doma  2022/5/10             1           -299 -200   
2  0442.HK_2022-05-10    Doma  2022/5/10             1           -298 -199   
3  0442.HK_2022-05-10    Doma  2022/5/10            

In [19]:
# 存储模型参数
model_params = {}

# 为每个公司估计模型
for ticker, firm_data in df.groupby('ticker'):
    # 提取估计窗口数据
    est_data = firm_data[firm_data['window'] == 'estimation'].copy()
    
    # 检查是否有足够数据
    if len(est_data) < 120:
        print(f"跳过公司 {ticker}: 估计窗口只有 {len(est_data)} 个观测值 (<120)")
        continue
    
    # 准备数据
    X_capm = est_data[['MKT_RF']].copy()
    
    # 根据因子存在性构建FF模型
    factor_cols = ['MKT_RF']
    if 'SMB' in est_data.columns and 'HML' in est_data.columns:
        factor_cols.extend(['SMB', 'HML'])
    if 'MOM' in est_data.columns:
        factor_cols.append('MOM')
    
    X_ff = est_data[factor_cols].copy()
    
    # 添加常数项
    X_capm = sm.add_constant(X_capm)
    X_ff = sm.add_constant(X_ff)
    
    y = est_data['excess_ret']
    
    try:
        # 估计CAPM模型
        model_capm = sm.OLS(y, X_capm, missing='drop').fit()
        
        # 估计FF模型
        model_ff = sm.OLS(y, X_ff, missing='drop').fit()
        
        # 保存模型参数
        model_params[ticker] = {
            'capm_alpha': model_capm.params.get('const', 0),
            'capm_beta_mkt': model_capm.params.get('MKT_RF', 0),
            'capm_resid_std': np.std(model_capm.resid),
            'ff_alpha': model_ff.params.get('const', 0),
            'ff_params': {col: model_ff.params.get(col, 0) for col in factor_cols},
            'est_window_len': len(est_data),
            'est_mkt_mean': est_data['MKT_RF'].mean(),
            'est_mkt_var': est_data['MKT_RF'].var()
        }
        
        # 如果是FF3/4模型，计算每个因子的beta
        for i, factor in enumerate(factor_cols, 1):
            model_params[ticker][f'ff_beta_{factor.lower()}'] = model_ff.params.get(factor, 0)
        
    except Exception as e:
        print(f"公司 {ticker} 模型估计失败: {e}")

print(f"\n成功估计模型的公司数量: {len(model_params)}")
print(f"被剔除的公司数量: {df['ticker'].nunique() - len(model_params)}")


成功估计模型的公司数量: 152
被剔除的公司数量: 0


In [20]:
# 筛选有效公司
valid_tickers = list(model_params.keys())
df_valid = df[df['ticker'].isin(valid_tickers)].copy()

# 为每家公司计算AR和CAR
for ticker, firm_data in df_valid.groupby('ticker'):
    if ticker not in model_params:
        continue
    
    params = model_params[ticker]
    
    # 为CAPM模型计算正常收益率
    normal_ret_capm = (
        params['capm_alpha'] + 
        params['capm_beta_mkt'] * firm_data['MKT_RF']
    )
    
    # 为FF模型计算正常收益率
    ff_factors = ['MKT_RF']
    if 'SMB' in firm_data.columns and 'HML' in firm_data.columns:
        ff_factors.extend(['SMB', 'HML'])
    if 'MOM' in firm_data.columns:
        ff_factors.append('MOM')
    
    normal_ret_ff = params['ff_alpha']
    for factor in ff_factors:
        param_name = f'ff_beta_{factor.lower()}'
        if param_name in params:
            normal_ret_ff += params[param_name] * firm_data[factor]
        else:
            # 如果缺少某个因子，检查是否为FF4但只有FF3的情况
            if factor in ['MOM'] and param_name not in params:
                # 假设缺少MOM因子，将其设为0
                normal_ret_ff += 0
            else:
                print(f"警告: 公司 {ticker} 缺少因子 {factor} 的参数")
    
    # 计算AR
    ar_capm = firm_data['ret'] - normal_ret_capm
    ar_ff = firm_data['ret'] - normal_ret_ff
    
    # 更新原始数据框
    df.loc[firm_data.index, 'AR_CAPM'] = ar_capm.values
    df.loc[firm_data.index, 'AR_FF4'] = ar_ff.values
    
    # 计算标准化AR（用于Patell检验）
    # 预测误差的方差调整
    est_len = params['est_window_len']
    mkt_diff = firm_data['MKT_RF'] - params['est_mkt_mean']
    mkt_var_sum = params['est_mkt_var'] * (est_len - 1)  # 方差 * (n-1)
    
    if params['est_mkt_var'] > 0 and est_len > 0:
        # CAPM模型的标准化
        prediction_var_capm = params['capm_resid_std']**2 * (
            1 + 1/est_len + (mkt_diff**2) / mkt_var_sum
        )
        ar_st_capm = ar_capm / np.sqrt(prediction_var_capm)
        
        # FF模型的标准化（简化处理，使用相同的残差标准差）
        ar_st_ff = ar_ff / np.sqrt(prediction_var_capm)
        
        df.loc[firm_data.index, 'AR_st_CAPM'] = ar_st_capm.values
        df.loc[firm_data.index, 'AR_st_FF4'] = ar_st_ff.values
    
    # 计算每个事件窗口的CAR
    for window_name, window_func in [
        ('short_term', lambda t: -1 <= t <= 1),
        ('medium_term', lambda t: 0 <= t <= 5),
        ('extended', lambda t: -5 <= t <= 30),
        ('post_event', lambda t: 0 <= t <= 60)
    ]:
        window_mask = firm_data['t'].apply(window_func)
        if window_mask.any():
            # 计算CAR
            car_capm = ar_capm[window_mask].sum()
            car_ff = ar_ff[window_mask].sum()
            
            # 更新到原表
            df.loc[firm_data.index, f'CAR_CAPM_{window_name}'] = car_capm
            df.loc[firm_data.index, f'CAR_FF4_{window_name}'] = car_ff

# 检查结果
print("\n计算完成后的数据列名:")
print([col for col in df.columns if 'AR_' in col or 'CAR_' in col])

print("\n前5行数据的AR和CAR:")
ar_car_cols = ['ticker', 't', 'ret', 'window', 'AR_CAPM', 'AR_FF4', 
               'CAR_CAPM_short_term', 'CAR_FF4_short_term']
print(df[ar_car_cols].head(10))

# 保存包含AR和CAR的数据
df.to_csv('最终面板数据_with_AR_CAR.csv', index=False)
print(f"\n数据已保存到: 最终面板数据_with_AR_CAR.csv")


计算完成后的数据列名:
['AR_CAPM', 'AR_FF4', 'CAR_CAPM_short_term', 'CAR_CAPM_medium_term', 'CAR_CAPM_extended', 'CAR_CAPM_post_event', 'CAR_FF4_short_term', 'CAR_FF4_medium_term', 'CAR_FF4_extended', 'CAR_FF4_post_event', 'AR_st_CAPM', 'AR_st_FF4']

前5行数据的AR和CAR:
    ticker    t       ret      window   AR_CAPM    AR_FF4  \
0  0442.HK -201       NaN  estimation       NaN       NaN   
1  0442.HK -200  0.000000  estimation  0.001303 -0.001061   
2  0442.HK -199  0.133531  estimation  0.135204  0.142372   
3  0442.HK -198  0.000000  estimation  0.002314  0.012722   
4  0442.HK -197  0.000000  estimation -0.000718 -0.004394   
5  0442.HK -196  0.000000  estimation  0.000011 -0.005123   
6  0442.HK -195  0.128833  estimation  0.129670  0.134432   
7  0442.HK -194 -0.068208  estimation -0.068236 -0.067855   
8  0442.HK -193  0.000000  estimation  0.000759 -0.005020   
9  0442.HK -192  0.000000  estimation  0.001566  0.000629   

   CAR_CAPM_short_term  CAR_FF4_short_term  
0             0.011756      

In [21]:
def calculate_p_value(t_stat, df):
    """计算双尾检验的p值并添加显著性标记"""
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))
    
    if p_value < 0.01:
        sig = '***'
    elif p_value < 0.05:
        sig = '**'
    elif p_value < 0.10:
        sig = '*'
    else:
        sig = ''
    
    return p_value, sig

def calculate_car_stats(window_name, model_prefix='CAPM'):
    """计算特定窗口的CAR统计量"""
    car_col = f'CAR_{model_prefix}_{window_name}'
    ar_st_col = f'AR_st_{model_prefix}'
    
    # 获取该窗口的唯一CAR值（每家公司一个）
    car_data = df.drop_duplicates(subset=['ticker', car_col])[[car_col, 'ticker']].dropna()
    
    if len(car_data) == 0:
        return None
    
    # 平均CAR
    avg_car = car_data[car_col].mean()
    n_firms = len(car_data)
    
    # 1. 简单的t检验
    t_stat_simple, p_val_simple = stats.ttest_1samp(car_data[car_col], 0)
    p_val_simple, sig_simple = calculate_p_value(t_stat_simple, n_firms-1)
    
    # 2. Patell检验（使用标准化AR之和）
    # 获取标准化AR并计算窗口内的标准化CAR
    patell_cars = []
    for ticker, firm_data in df[df['window'] == window_name].groupby('ticker'):
        if ar_st_col in firm_data.columns:
            car_st = firm_data[ar_st_col].sum()
            patell_cars.append(car_st)
    
    if patell_cars:
        t_stat_patell, p_val_patell = stats.ttest_1samp(patell_cars, 0)
        p_val_patell, sig_patell = calculate_p_value(t_stat_patell, len(patell_cars)-1)
    else:
        t_stat_patell = p_val_patell = np.nan
        sig_patell = ''
    
    # 3. BMP检验
    if patell_cars:
        # BMP统计量
        bmp_stat = np.mean(patell_cars) / (np.std(patell_cars, ddof=1) / np.sqrt(len(patell_cars)))
        p_val_bmp, sig_bmp = calculate_p_value(bmp_stat, len(patell_cars)-1)
    else:
        bmp_stat = p_val_bmp = np.nan
        sig_bmp = ''
    
    # 4. Corrado秩检验
    # 收集所有AR（估计窗口+事件窗口）
    all_ars = []
    for ticker, firm_data in df.groupby('ticker'):
        if f'AR_{model_prefix}' in firm_data.columns:
            all_ars.extend(firm_data[f'AR_{model_prefix}'].dropna().tolist())
    
    if all_ars:
        # 为所有AR分配排名
        ranks = stats.rankdata(all_ars)
        
        # 提取事件窗口的AR排名
        event_ars = []
        for ticker, firm_data in df[df['window'] == window_name].groupby('ticker'):
            if f'AR_{model_prefix}' in firm_data.columns:
                event_ars.extend(firm_data[f'AR_{model_prefix}'].tolist())
        
        if event_ars:
            # 获取事件AR在总AR中的排名
            event_indices = [i for i, ar in enumerate(all_ars) if ar in event_ars]
            event_ranks = [ranks[i] for i in event_indices if i < len(ranks)]
            
            if event_ranks:
                # Corrado检验
                K_bar = np.mean(event_ranks)
                N = len(all_ars)
                expected_K = (N + 1) / 2
                rank_std = np.sqrt(np.sum((ranks - expected_K)**2) / (N - 1))
                t_corrado = (K_bar - expected_K) / (rank_std / np.sqrt(len(event_ranks)))
                p_val_corrado, sig_corrado = calculate_p_value(t_corrado, len(event_ranks)-1)
            else:
                t_corrado = p_val_corrado = np.nan
                sig_corrado = ''
        else:
            t_corrado = p_val_corrado = np.nan
            sig_corrado = ''
    else:
        t_corrado = p_val_corrado = np.nan
        sig_corrado = ''
    
    return {
        'window': window_name,
        'model': model_prefix,
        'N': n_firms,
        'avg_CAR': avg_car,
        't_simple': t_stat_simple,
        'p_simple': p_val_simple,
        'sig_simple': sig_simple,
        't_patell': t_stat_patell,
        'p_patell': p_val_patell,
        'sig_patell': sig_patell,
        't_bmp': bmp_stat,
        'p_bmp': p_val_bmp,
        'sig_bmp': sig_bmp,
        't_corrado': t_corrado,
        'p_corrado': p_val_corrado,
        'sig_corrado': sig_corrado
    }

# 为每个窗口和模型计算统计量
windows_to_test = ['short_term', 'medium_term', 'extended', 'post_event']
models_to_test = ['CAPM', 'FF4']

all_results = []
for window in windows_to_test:
    for model in models_to_test:
        result = calculate_car_stats(window, model)
        if result:
            all_results.append(result)

# 创建结果DataFrame
results_df = pd.DataFrame(all_results)

# 格式化输出结果
print("=" * 120)
print("事件研究结果汇总")
print("=" * 120)

# 按模型分组显示
for model in models_to_test:
    model_results = results_df[results_df['model'] == model]
    if len(model_results) > 0:
        print(f"\n{'='*60}")
        print(f"模型: {model}")
        print(f"{'='*60}")
        
        for _, row in model_results.iterrows():
            print(f"\n窗口: {row['window']} (N={int(row['N']):,})")
            print(f"平均CAR: {row['avg_CAR']:.6f}")
            print(f"简单t检验: t={row['t_simple']:.4f}, p={row['p_simple']:.4f}{row['sig_simple']}")
            print(f"Patell检验: t={row['t_patell']:.4f}, p={row['p_patell']:.4f}{row['sig_patell']}")
            print(f"BMP检验:   t={row['t_bmp']:.4f}, p={row['p_bmp']:.4f}{row['sig_bmp']}")
            print(f"Corrado检验: t={row['t_corrado']:.4f}, p={row['p_corrado']:.4f}{row['sig_corrado']}")

# 保存结果
results_df.to_csv('三个检验的显著性报告.csv', index=False)
print(f"\n统计检验结果已保存到: event_study_test_results.csv")

事件研究结果汇总

模型: CAPM

窗口: short_term (N=152)
平均CAR: 0.100296
简单t检验: t=6.3339, p=0.0000***
Patell检验: t=8.9767, p=0.0000***
BMP检验:   t=8.9767, p=0.0000***
Corrado检验: t=-1.0959, p=0.2733

窗口: medium_term (N=152)
平均CAR: 0.220873
简单t检验: t=11.9446, p=0.0000***
Patell检验: t=12.1105, p=0.0000***
BMP检验:   t=12.1105, p=0.0000***
Corrado检验: t=-0.1652, p=0.8688

窗口: extended (N=152)
平均CAR: 1.307346
简单t检验: t=19.3889, p=0.0000***
Patell检验: t=14.2049, p=0.0000***
BMP检验:   t=14.2049, p=0.0000***
Corrado检验: t=-0.7592, p=0.4478

窗口: post_event (N=152)
平均CAR: 2.194621
简单t检验: t=21.9846, p=0.0000***
Patell检验: t=13.9686, p=0.0000***
BMP检验:   t=13.9686, p=0.0000***
Corrado检验: t=0.3737, p=0.7086

模型: FF4

窗口: short_term (N=152)
平均CAR: 0.097543
简单t检验: t=6.4635, p=0.0000***
Patell检验: t=8.9817, p=0.0000***
BMP检验:   t=8.9817, p=0.0000***
Corrado检验: t=-1.3218, p=0.1864

窗口: medium_term (N=152)
平均CAR: 0.218306
简单t检验: t=12.1212, p=0.0000***
Patell检验: t=12.2239, p=0.0000***
BMP检验:   t=12.2239, p=0.0000***
Corrado检验: t=-